In [2]:
from huggingface_hub import login

with open("hf_token") as file:
    token = file.readline().strip()
    login(token)

In [ ]:
import sounddevice as sd
import librosa
import transformers
import warnings
warnings.filterwarnings('ignore')

pipe = transformers.pipeline(model='fixie-ai/ultravox-v0_5-llama-3_2-1b', trust_remote_code=True)

def record_audio(duration=5, fs=16000):
    print(f"Recording for {duration} seconds...")
    audio = sd.rec(int(duration * fs), samplerate=fs, channels=1, dtype='float32')
    sd.wait()
    print("Recording finished.")
    return audio.flatten(), fs

In [17]:
turns = [
    {
        "role": "system",
        "content": "You are a friendly and helpful character. You love to answer questions for people."
    }
]

while True:
    # Record audio
    raw_audio, _ = record_audio(duration=5, fs=16000)

    # Resample audio with librosa (ensures 16kHz mono)
    audio = librosa.resample(raw_audio, orig_sr=16000, target_sr=16000)

    # Pass audio and history to the model
    output = pipe({'audio': audio, 'turns': turns, 'sampling_rate': 16000}, max_new_tokens=30)
    assistant_reply = output

    print(f"\n🧠 Assistant: {assistant_reply}")

    # Clean out old user audio messages (only keep one <audio>)
    turns = [t for t in turns if not (t["role"] == "user" and "audio" in t["content"])]

    # Append latest turn
    turns.append({"role": "assistant", "content": assistant_reply})

    print("\n--- Speak again or Ctrl+C to exit ---")

Recording for 5 seconds...
Recording finished.

🧠 Assistant: I'm glad you're hoping it works out okay. However, I need to clarify that this conversation just started, and we haven't discussed anything yet

--- Speak again or Ctrl+C to exit ---
Recording for 5 seconds...
Recording finished.

🧠 Assistant: No, I don't think so. This is the beginning of our conversation, and we haven't discussed any specific topic or question yet. I'm

--- Speak again or Ctrl+C to exit ---
Recording for 5 seconds...
Recording finished.

🧠 Assistant: It seems like you're starting with a phrase in a foreign language. "Ah" is a common way to start sentences in many languages, including Spanish

--- Speak again or Ctrl+C to exit ---
Recording for 5 seconds...
Recording finished.

🧠 Assistant: When using an audio player or media player, such as VLC or Windows Media Player, you can often add an audio placeholder, also known as a "

--- Speak again or Ctrl+C to exit ---
Recording for 5 seconds...
Recording finish

KeyboardInterrupt: 